In [0]:
from pyspark.sql import functions as F, Window


greedy_output = "dbfs:/Volumes/lsg/pre_dm/tfsdl_lsg_ops_prod/inventory_rebalance/greedy_output"

optimized_df = spark.read.parquet(greedy_output)


lane_candidates_analysis_dir = "dbfs:/Volumes/lsg/pre_dm/tfsdl_lsg_ops_prod/inventory_rebalance/lane_candidates_analysis_df"
source_analysis_df  = spark.read.parquet(lane_candidates_analysis_dir)

In [0]:
print(source_analysis_df.columns)

In [0]:
optimized_analysis_df = (
    source_analysis_df.alias("s")
    .join(
        optimized_df.alias("o"),
        on=[
            F.col("sku_number") == F.col("source_sku_number"),
            F.col("source_branch") == F.col("source_branch_code"),
            F.col("lot_number") == F.col("source_lot_number"),
            F.col("inventory_location_code") == F.col("source_inventory_location_code"),
            F.col("destination_branch") == F.col("destination_Branch_Code")
        ],
        how="left"
    )
)

sku_branch_window = Window.partitionBy(
    "source_sku_number",
    "source_branch_code"
)

optimized_analysis_df = (
    optimized_analysis_df
    .withColumn(
        "optimized_row_count_by_sku_branch",
        F.count("optimized_move_qty").over(sku_branch_window)
    )
    .filter(F.col("optimized_row_count_by_sku_branch") > 0)
)


In [0]:
display(optimized_analysis_df)